# Preprocesamiento y Embeddings
## Motor de búsqueda semántica de ofertas laborales

**Proyecto Integrador 1 — Ingeniería de Sistemas**

### Objetivo del notebook

Continuar el flujo de trabajo a partir de las conclusiones del EDA (`01_EDA.ipynb`), cubriendo los tres puntos indicados por el tutor:

1. **Terminar la limpieza y el preprocesamiento** de los datos.
2. **Primera iteración de representación semántica** mediante *Sentence-BERT* (Sentence Transformers).
3. **Exploración de bases de datos vectoriales** (Pinecone y alternativas), comparándolas con FAISS (usado en el anteproyecto).

Este notebook asume que `01_EDA.ipynb` ya se ejecutó y que el dataset crudo está disponible en `/data/raw/job_descriptions.csv` (o se vuelve a descargar de Kaggle).

## 1. Importación de librerías

Para esta etapa se necesitan, además de Pandas/NumPy:

- **sentence-transformers**: para generar los embeddings (Sentence-BERT).
- **faiss-cpu**: índice vectorial local, usado como línea base (baseline) — es lo que se propuso en el anteproyecto.
- **re**: limpieza de texto con expresiones regulares.
- **tqdm**: barra de progreso al generar embeddings sobre muchos registros.

Instalación (en Colab, descomentar la primera vez):

In [ ]:
# !pip install -q sentence-transformers faiss-cpu tqdm pinecone-client chromadb


In [ ]:
import os
import re
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 2. Carga del dataset

**Importante en Colab:** el almacenamiento local (`/content/...`) es efímero — se borra cada vez que se reinicia o desconecta el runtime. Como este notebook suele abrirse en una sesión distinta a la del EDA, no se puede asumir que el CSV ya está en disco.

Para no tener que descargar 1.6M de registros (y volver a correr toda la limpieza) cada vez que se abre una sesión nueva, se monta Google Drive y se persisten ahí tanto el dataset crudo como los archivos procesados. Si el archivo ya existe en Drive (por ejemplo porque ya corriste este notebook antes o porque un compañero de equipo ya lo dejó ahí), se reutiliza; si no, se descarga de Kaggle.

In [ ]:
from pathlib import Path

MONTAR_DRIVE = True  # cambia a False si prefieres trabajar solo con el almacenamiento efimero de Colab

if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/proyecto_integrador")
else:
    BASE_DIR = Path("/content")

RUTA_RAW = BASE_DIR / "data" / "raw"
RUTA_PROCESSED = BASE_DIR / "data" / "processed"
RUTA_RAW.mkdir(parents=True, exist_ok=True)
RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

RUTA_DATASET = RUTA_RAW / "job_descriptions.csv"
print(f"Dataset crudo esperado en: {RUTA_DATASET}")
print(f"Archivos procesados se guardaran en: {RUTA_PROCESSED}")


In [ ]:
if not RUTA_DATASET.exists():
    print("Dataset no encontrado, descargando desde Kaggle...")
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !kaggle datasets download -d ravindrasinghrana/job-description-dataset -p {RUTA_RAW} --unzip
else:
    print(f"Dataset ya existe en {RUTA_DATASET}, se omite la descarga.")


In [ ]:
df = pd.read_csv(RUTA_DATASET)
print(f"Registros cargados: {len(df):,}")
df.head(2)


## 3. Limpieza y preprocesamiento

A partir de las conclusiones del EDA, se aplican las siguientes decisiones:

| Hallazgo del EDA | Decisión de limpieza |
|---|---|
| `Company Profile` tiene 0.34% de nulos | Se rellenan con cadena vacía (no se usará para embeddings de todas formas) |
| Columnas de texto (`Job Title`, `Role`, `Qualifications`, `skills`, `Responsibilities`, `Job Description`) sin nulos relevantes, pero pueden tener espacios/ruido | Normalizar espacios en blanco, quitar saltos de línea repetidos |
| `Experience` es texto tipo `"5 to 8 Years"` | Extraer `experiencia_min` y `experiencia_max` numéricos para poder filtrar |
| `Salary Range` es texto tipo `"$59K-$99K"` | Extraer `salario_min` y `salario_max` numéricos (en miles) |
| `Job Posting Date` ya se parseó como fecha en el EDA | Se repite aquí por si se ejecuta el notebook de forma independiente |
| Columnas irrelevantes para búsqueda semántica (`Contact Person`, `Contact`, `Job Portal`) | Se descartan del dataset que alimentará el motor (se conservan solo si se necesitan para trazabilidad) |
| 0 registros duplicados, `Job Id` 100% único | No se requiere deduplicación adicional |


In [ ]:
COLUMNAS_TEXTO = [
    "Job Title",
    "Role",
    "Qualifications",
    "skills",
    "Responsibilities",
    "Job Description",
]

COLUMNAS_FILTROS = [
    "Experience",
    "Salary Range",
    "location",
    "Country",
    "Work Type",
    "Company Size",
    "Preference",
    "Job Posting Date",
]

COLUMNAS_IDENTIFICACION = ["Job Id", "Company"]

def limpiar_texto(texto: str) -> str:
    """Normaliza espacios en blanco y remueve caracteres de control."""
    if pd.isna(texto):
        return ""
    texto = str(texto)
    texto = re.sub(r"\s+", " ", texto)  # colapsa espacios/saltos de línea
    return texto.strip()

df_clean = df.copy()

for columna in COLUMNAS_TEXTO:
    df_clean[columna] = df_clean[columna].apply(limpiar_texto)

df_clean["Company Profile"] = df_clean["Company Profile"].fillna("")

print("Columnas de texto normalizadas.")


In [ ]:
# Parseo de Experience: "5 to 8 Years" -> experiencia_min=5, experiencia_max=8
# Version vectorizada (str.extract), mucho mas rapida que .apply() fila por fila sobre 1.6M registros.
extraido_experiencia = df_clean["Experience"].str.extract(
    r"(\d+)\s*to\s*(\d+)", flags=re.IGNORECASE
)

df_clean["experiencia_min"] = pd.to_numeric(extraido_experiencia[0])
df_clean["experiencia_max"] = pd.to_numeric(extraido_experiencia[1])

df_clean[["Experience", "experiencia_min", "experiencia_max"]].head()


In [ ]:
# Parseo de Salary Range: "$59K-$99K" -> salario_min=59, salario_max=99 (en miles USD)
# Version vectorizada, igual que con Experience.
extraido_salario = df_clean["Salary Range"].str.extract(
    r"\$?(\d+)K-\$?(\d+)K", flags=re.IGNORECASE
)

df_clean["salario_min"] = pd.to_numeric(extraido_salario[0])
df_clean["salario_max"] = pd.to_numeric(extraido_salario[1])

df_clean[["Salary Range", "salario_min", "salario_max"]].head()


In [ ]:
df_clean["Job Posting Date"] = pd.to_datetime(df_clean["Job Posting Date"], errors="coerce")

columnas_a_descartar = [c for c in ["Contact Person", "Contact", "Job Portal"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=columnas_a_descartar)

print(f"Columnas descartadas: {columnas_a_descartar}")
print(f"Dimensiones tras limpieza: {df_clean.shape}")


### 3.1 Construcción del texto combinado para el embedding

Se concatenan los campos textuales relevantes en un único campo `texto_combinado`. Se repite `Job Title` y `skills` para darles un poco más de peso semántico, ya que suelen ser los campos más discriminativos para una búsqueda por habilidades/cargo.

In [ ]:
# Version vectorizada (str.cat), en lugar de .apply(axis=1) que es muy lento sobre 1.6M filas.
partes = [
    df_clean["Job Title"], df_clean["Job Title"],   # peso extra
    df_clean["Role"],
    df_clean["skills"], df_clean["skills"],          # peso extra
    df_clean["Qualifications"],
    df_clean["Responsibilities"],
    df_clean["Job Description"],
]

df_clean["texto_combinado"] = partes[0].str.cat(partes[1:], sep=" ").str.strip()

df_clean[["Job Title", "texto_combinado"]].head(2)


In [ ]:
# Se descartan registros sin ningún contenido textual útil (deberían ser 0 según el EDA, se valida igual)
antes = len(df_clean)
df_clean = df_clean[df_clean["texto_combinado"].str.len() > 0].reset_index(drop=True)
print(f"Registros removidos por texto vacío: {antes - len(df_clean):,}")
print(f"Registros finales: {len(df_clean):,}")


In [ ]:
RUTA_LIMPIO = RUTA_PROCESSED / "job_descriptions_clean.parquet"
df_clean.to_parquet(RUTA_LIMPIO, index=False)
print(f"Dataset limpio guardado en: {RUTA_LIMPIO}")


## 4. Primera iteración de embeddings con Sentence-BERT

Con 1,615,940 registros, generar embeddings de **todo** el dataset en un notebook de Colab puede tardar horas incluso con GPU (según el modelo). Para esta primera iteración:

- Se trabaja sobre una **muestra representativa** (configurable, por defecto 50,000 registros) para validar el pipeline end-to-end rápidamente.
- Se usa `all-MiniLM-L6-v2`: modelo pequeño (384 dimensiones), rápido, buen punto de partida para *semantic search*. Si el desempeño no es suficiente en la fase de evaluación, se puede migrar a `all-mpnet-base-v2` (mejor calidad, más lento) en una segunda iteración.
- El embedding completo del dataset se puede ejecutar después en un job por lotes (batch) una vez validado el pipeline, idealmente con GPU.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "all-MiniLM-L6-v2"
TAMANO_MUESTRA_EMBEDDINGS = 50_000  # ajustar según recursos disponibles

modelo = SentenceTransformer(MODELO_EMBEDDINGS)
print(f"Modelo cargado: {MODELO_EMBEDDINGS}")
print(f"Dimensión del embedding: {modelo.get_sentence_embedding_dimension()}")


In [ ]:
muestra_embeddings = df_clean.sample(
    n=min(TAMANO_MUESTRA_EMBEDDINGS, len(df_clean)),
    random_state=42,
).reset_index(drop=True)

print(f"Registros a codificar: {len(muestra_embeddings):,}")


In [ ]:
inicio = time.time()

embeddings = modelo.encode(
    muestra_embeddings["texto_combinado"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # normaliza a norma 1 -> producto punto == similitud coseno
)

duracion = time.time() - inicio
print(f"Embeddings generados: {embeddings.shape}")
print(f"Tiempo total: {duracion:.1f} s  ({duracion / len(muestra_embeddings) * 1000:.2f} ms/registro)")


In [ ]:
np.save(RUTA_PROCESSED / "embeddings_muestra.npy", embeddings)
muestra_embeddings.to_parquet(RUTA_PROCESSED / "muestra_embeddings_meta.parquet", index=False)
print("Embeddings y metadatos guardados en Drive (persisten entre sesiones de Colab).")


### 4.1 Prueba rápida de búsqueda (baseline con FAISS)

Antes de explorar bases de datos vectoriales externas, se valida el pipeline con **FAISS** (tal como se planteó en el anteproyecto), usando un índice plano (`IndexFlatIP`) sobre la muestra.

In [ ]:
import faiss

dimension = embeddings.shape[1]
indice_faiss = faiss.IndexFlatIP(dimension)  # producto interno == coseno, porque los vectores están normalizados
indice_faiss.add(embeddings)

print(f"Vectores indexados en FAISS: {indice_faiss.ntotal:,}")


In [ ]:
def buscar_ofertas(consulta: str, k: int = 5):
    vector_consulta = modelo.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    similitudes, indices = indice_faiss.search(vector_consulta, k)

    resultados = muestra_embeddings.iloc[indices[0]].copy()
    resultados["similitud"] = similitudes[0]
    return resultados[["Job Title", "Role", "Country", "Work Type", "similitud"]]

# Ejemplo
buscar_ofertas("python developer with machine learning and NLP experience", k=5)


## 5. Exploración de bases de datos vectoriales

El anteproyecto planteó **FAISS** como motor de búsqueda vectorial. FAISS es una biblioteca (no una base de datos): es muy rápida y gratuita, pero corre en memoria/proceso local, no maneja persistencia ni actualizaciones incrementales de forma nativa, y no está pensada como servicio con API propia. Para la etapa de integración (API + interfaz web) conviene comparar alternativas de **bases de datos vectoriales** administradas o auto-hospedadas.

| Opción | Tipo | Costo | Persistencia / escalabilidad | Filtrado por metadatos | Curva de aprendizaje | Cuándo conviene |
|---|---|---|---|---|---|---|
| **FAISS** | Biblioteca en memoria | Gratis | Manual (hay que guardar/cargar el índice); escala vertical | Limitado, se maneja aparte en Pandas/SQL | Baja | Prototipos, datasets que caben en memoria/disco de un solo servidor |
| **Chroma** | Base de datos vectorial embebida/self-hosted | Gratis (open source) | Persistencia en disco nativa; fácil de correr local o en Docker | Sí, nativo | Baja | Buen punto medio para pasar de FAISS a algo con persistencia sin salir de Python |
| **Qdrant** | Base de datos vectorial self-hosted o cloud (tiene *free tier*) | Gratis (self-host) / plan gratuito cloud limitado | Persistencia, filtrado avanzado, buena para producción | Sí, muy completo | Media | Cuando se quiere una API HTTP/gRPC propia y filtros complejos sobre metadatos |
| **Pinecone** | Base de datos vectorial 100% administrada (SaaS) | Plan gratuito limitado (una región, límite de vectores); pago por uso a partir de cierto volumen | Totalmente administrada, escala automáticamente | Sí | Baja (API simple) | Cuando no se quiere administrar infraestructura y se prioriza velocidad de desarrollo |
| **Weaviate** | Base de datos vectorial self-hosted o cloud | Gratis (self-host) / cloud con costo | Persistencia, soporta búsqueda híbrida (vectorial + palabras clave) | Sí | Media-alta | Si además de similitud semántica se quiere combinar con búsqueda léxica (BM25) |

**Recomendación para esta iteración:** dado que el equipo ya validó FAISS en el anteproyecto y el presupuesto no contempla servicios pagos recurrentes más allá de lo presupuestado, una ruta razonable es:

1. Mantener **FAISS** como baseline de referencia (ya funciona, es gratis, cero dependencias externas).
2. Probar **Pinecone** en su capa gratuita para evaluar si la facilidad de integración (API lista, sin mantener infraestructura) compensa el límite de vectores del plan free — es útil para la demo/API del alcance del proyecto.
3. Si el plan gratuito de Pinecone resulta insuficiente para 1.6M de registros, evaluar **Qdrant** o **Chroma** self-hosted en el mismo VPS ya presupuestado (Railway), que no tienen límite artificial de vectores.

A continuación, un ejemplo mínimo (no ejecutado, requiere API key) de cómo se vería la integración con Pinecone para comparar el mismo flujo de búsqueda.

In [ ]:
# Ejemplo ilustrativo de integración con Pinecone (requiere cuenta y API key gratuitas).
# No se ejecuta aquí porque necesita credenciales.
#
# from pinecone import Pinecone, ServerlessSpec
#
# pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
#
# NOMBRE_INDICE = "ofertas-laborales"
# if NOMBRE_INDICE not in [i.name for i in pc.list_indexes()]:
#     pc.create_index(
#         name=NOMBRE_INDICE,
#         dimension=dimension,          # 384 para all-MiniLM-L6-v2
#         metric="cosine",
#         spec=ServerlessSpec(cloud="aws", region="us-east-1"),
#     )
#
# indice_pinecone = pc.Index(NOMBRE_INDICE)
#
# vectores = [
#     (str(row["Job Id"]), emb.tolist(), {"Job Title": row["Job Title"], "Country": row["Country"]})
#     for row, emb in zip(muestra_embeddings.to_dict("records"), embeddings)
# ]
# for lote in range(0, len(vectores), 100):
#     indice_pinecone.upsert(vectores[lote:lote + 100])
#
# resultado = indice_pinecone.query(
#     vector=modelo.encode("python developer with machine learning").tolist(),
#     top_k=5,
#     include_metadata=True,
# )


In [ ]:
# Ejemplo ilustrativo de integración con Chroma (sí se puede instalar localmente y probar sin API key externa).
# import chromadb
#
# cliente_chroma = chromadb.PersistentClient(path="str(RUTA_PROCESSED / "chroma_db")")
# coleccion = cliente_chroma.get_or_create_collection(name="ofertas_laborales", metadata={"hnsw:space": "cosine"})
#
# coleccion.add(
#     ids=[str(i) for i in muestra_embeddings["Job Id"]],
#     embeddings=embeddings.tolist(),
#     metadatas=muestra_embeddings[["Job Title", "Role", "Country", "Work Type"]].to_dict("records"),
# )
#
# resultados_chroma = coleccion.query(
#     query_embeddings=modelo.encode(["python developer with machine learning"]).tolist(),
#     n_results=5,
# )


## 6. Próximos pasos

1. **Decidir la base de datos vectorial** definitiva a partir de esta comparación (recomendado: probar Pinecone free tier + Chroma self-hosted, y quedarse con el que mejor equilibre costo/latencia/facilidad de integración con la API).
2. **Escalar el embedding** al dataset completo (o a un subconjunto representativo mayor) una vez elegida la base de datos, idealmente en un proceso por lotes fuera del notebook.
3. Evaluar si `all-MiniLM-L6-v2` es suficiente o si conviene migrar a un modelo más grande (`all-mpnet-base-v2`) según los resultados de la evaluación cualitativa de consultas de prueba.
4. Con la base vectorial elegida, comenzar el **Objetivo específico 4**: desarrollo de la API y la interfaz web.